# queue

> at-least-once work in the vault: retries with backoff, leases that survive a crash, and a dead letter

In [ ]:
#| default_exp queue

In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.test import *

`enqueue` writes a row. `claim` takes it with one `UPDATE ... RETURNING` through a partial index,
so two workers racing get disjoint sets. A worker that dies holds a lease that expires, and
`reclaim` puts the job back. Failures back off and dead-letter once the attempts are spent.

This is vishalakshi's queue, moved here for ramabana and vishalakshi to share. Two things are
new. `job_runs` now keeps what the handler returned, which turns "show me the trend" into a query
over past runs rather than a fresh walk. And both tables come from `Store`.

There is no daemon. `drain` is the tick, and a cron entry, an app thread or a button calls it.

In [ ]:
#| export
import random, time
from fastcore.all import AttrDict, L, first, ifnone, patch, store_attr, xdumps
from pobblebonk.core import BUSY_TIMEOUT_MS, Store, clip, mkid, now, row, rows

In [ ]:
#| export
STATES = ('ready', 'running', 'done', 'dead')
MAX_ERR = 500      # chars of an error kept on the job row
MIN_RETRY = 1.     # seconds a retry is always pushed out by, so one `drain` cannot spend every attempt

In [ ]:
#| export
class Retry(Exception):
    'Raise from a handler to retry work that did not raise on its own: a bot wall, a 429, an empty read.'
    def __init__(self, msg:str='', after:float=None):
        super().__init__(msg)
        self.after = after   # seconds until the retry; None uses the backoff schedule

def backoff(attempt:int,       # failures so far; 1 on the first
            base:float=30,     # first delay, seconds
            cap:float=21600,   # longest delay, 6h
            jitter:float=0.25  # +/- fraction, so jobs that failed together do not retry together
) -> float:
    'Exponential delay before the next attempt.'
    return max(1., min(cap, base * 2**(attempt-1)) * (1 + random.uniform(-jitter, jitter)))

In [ ]:
test_eq(backoff(1, base=10, jitter=0) , 10)
test_eq(backoff(3, base=10, jitter=0) , 40)
test_eq(backoff(99, base=10, cap=100, jitter=0), 100)   # capped, not doubling forever

## The two tables

In [ ]:
#| export
class Runs(Store):
    "What each attempt did, and what the handler gave back. A fire's answer is kept here."
    name = 'job_runs'
    cols = dict(id=str, job_id=str, kind=str, worker=str, status=str, error=str, result=str,
                started_at=float, ended_at=float, took=float)
    idx = ('CREATE INDEX IF NOT EXISTS job_runs_job ON job_runs(job_id, started_at)',)

class Queue(Store):
    "An at-least-once job queue in the vault."
    name = 'jobs'
    cols = dict(id=str, kind=str, payload=str, key=str, state=str, priority=int, run_at=float,
                lease_until=float, worker=str, attempt=int, max_attempts=int, error=str,
                created_at=float, updated_at=float)
    idx = ("CREATE INDEX IF NOT EXISTS jobs_due ON jobs(priority, run_at) WHERE state='ready'",
           "CREATE INDEX IF NOT EXISTS jobs_lease ON jobs(lease_until) WHERE state='running'",
           # what stops two ticks enqueueing one schedule fire twice
           "CREATE UNIQUE INDEX IF NOT EXISTS jobs_key ON jobs(kind, key) "
           "WHERE key IS NOT NULL AND state IN ('ready','running')")

    def __init__(self,
                 db,                            # the vault
                 lease:float=600,               # seconds a claim is held before another worker may take it
                 max_attempts:int=5,            # attempts before a job is dead-lettered
                 base:float=30,                 # first retry delay; doubles per attempt
                 keep:float=604800,             # how long finished jobs survive `purge`, 7d
                 busy_ms:int=BUSY_TIMEOUT_MS):  # lock wait before a write gives up
        super().__init__(db, busy_ms)
        store_attr('lease,max_attempts,base,keep')
        self.runs, self.handlers = Runs(db, busy_ms), {}

    def __repr__(self):
        s = self.stats()
        return f"Queue({s['ready']} ready, {s['running']} running, {s['done']} done, {s['dead']} dead)"

## Putting work in

In [ ]:
#| export
@patch
def enqueue(self:Queue,
            kind:str,             # which handler runs it
            payload:dict=None,    # arguments, stored as JSON
            run_at:float=None,    # epoch seconds; now when unset
            priority:int=0,       # lower runs first
            key:str=None,         # idempotency key: an identical key still ready or running wins
            max_attempts:int=None,
            at:float=None         # when this is happening; now when unset
) -> AttrDict:
    'Add one job, or give back the pending one that `key` already names.'
    at = ifnone(at, now())
    r = dict(id=mkid(), kind=kind, payload=xdumps(payload or {}), key=key, state='ready',
             priority=priority, run_at=ifnone(run_at, at), lease_until=0., worker=None, attempt=0,
             max_attempts=max_attempts or self.max_attempts, error=None, created_at=at, updated_at=at)
    with self.txn():
        if key is not None and (was := self.one(
                "SELECT * FROM jobs WHERE kind=? AND key=? AND state IN ('ready','running')",
                [kind, key])): return was
        self.t.insert(r)
    return row(r)

@patch
def enqueue_all(self:Queue, kind:str, payloads, **kw) -> L:
    'Enqueue many jobs of one kind in a single transaction.'
    with self.txn(): return L(payloads).map(lambda p: self.enqueue(kind, p, **kw))

## Taking work out

`claim` picks *which* rows with `ORDER BY` and gets them back from `RETURNING` in no defined
order, so it sorts again before answering.

In [ ]:
#| export
@patch
def claim(self:Queue,
          worker:str='worker',  # who holds the lease; recorded on the row and in history
          n:int=1,              # how many jobs to take
          at:float=None
) -> L:
    'Take up to `n` due jobs, atomically.'
    at = ifnone(at, now())
    with self.txn():
        got = self.db.q(
            'UPDATE jobs SET state=?, worker=?, lease_until=?, updated_at=? WHERE id IN '
            '(SELECT id FROM jobs WHERE state=? AND run_at<=? ORDER BY priority, run_at LIMIT ?) '
            'RETURNING *', ['running', worker, at + self.lease, at, 'ready', at, n])
    return rows(sorted(got, key=lambda r: (r['priority'], r['run_at'])))

@patch
def _held(self:Queue, job) -> dict:
    "The row as it stands, or None when this worker no longer holds it. Call inside a `txn`."
    cur = self.one('SELECT * FROM jobs WHERE id=?', [job['id']])
    if cur is None or cur['state'] != 'running' or cur['worker'] != job.get('worker'): return None
    return cur

@patch
def ack(self:Queue, job, at:float=None) -> AttrDict:
    "Mark a claimed job finished. A worker whose lease was reclaimed cannot: it comes back `lost`."
    at = ifnone(at, now())
    with self.txn():
        if self._held(job) is None: return row(dict(job, state='lost'))
        self.db.q('UPDATE jobs SET state=?, error=NULL, worker=NULL, lease_until=0, updated_at=? '
                  'WHERE id=?', ['done', at, job['id']])
    return row(dict(job, state='done'))

@patch
def fail(self:Queue,
         job,               # the claimed job
         error:str,         # what went wrong
         after:float=None,  # seconds until the retry; None uses `backoff`
         at:float=None
) -> AttrDict:
    'Back off and retry, or dead-letter once the attempts are spent.'
    at, err = ifnone(at, now()), clip(error, MAX_ERR)
    with self.txn():
        if (cur := self._held(job)) is None: return row(dict(job, state='lost'))
        att = cur['attempt'] + 1
        dead = att >= (cur['max_attempts'] or self.max_attempts)
        # `backoff` never answers zero, and an `after` of 0 must not either: a job due again in
        # the same instant is re-claimed by the same `drain`, which spends five attempts at once
        wait = max(ifnone(after, backoff(att, base=self.base)), MIN_RETRY)
        nxt = cur['run_at'] if dead else at + wait
        self.db.q('UPDATE jobs SET state=?, attempt=?, run_at=?, error=?, worker=NULL, lease_until=0, '
                  'updated_at=? WHERE id=?',
                  ['dead' if dead else 'ready', att, nxt, err, at, cur['id']])
    return row(dict(cur, state='dead' if dead else 'ready', attempt=att, run_at=nxt, error=err))

@patch
def reclaim(self:Queue, at:float=None) -> L:
    'Give back jobs whose lease expired: a worker killed mid-job loses its claim, not its job.'
    at = ifnone(at, now())
    exp = self.q('SELECT * FROM jobs WHERE state=? AND lease_until>0 AND lease_until<=?',
                 ['running', at])
    # two pollers can read the same expired row; the fence in `fail` means only one of them counts it
    return exp.map(lambda j: self.fail(j, 'lease expired', at=at)).filter(lambda j: j['state'] != 'lost')

## Handlers, and the tick

A handler takes the payload and gives back anything JSON can hold. That answer is kept in
`job_runs.result`, which is the whole reason a daily roundup can talk about last week.

In [ ]:
#| export
@patch
def on(self:Queue, kind:str):
    'Decorator registering the handler for `kind`.'
    def _f(fn):
        self.handlers[kind] = fn
        return fn
    return _f

@patch
def register(self:Queue, kind:str, fn):
    'Register the handler for `kind`.'
    self.handlers[kind] = fn
    return fn

@patch
def _ran(self:Queue, job, status:str, t0:float, error:str=None, result=None) -> AttrDict:
    'Write one row of run history, keeping what the handler gave back, and report it.'
    at = now()
    self.runs.t.insert(dict(id=mkid(), job_id=job['id'], kind=job['kind'], worker=job.get('worker'),
                            status=status, error=(clip(error, MAX_ERR) or None), result=xdumps(result),
                            started_at=t0, ended_at=at, took=round(at-t0, 3)))
    return AttrDict(job_id=job['id'], kind=job['kind'], status=status, took=round(at-t0, 3),
                    error=error, result=result)

@patch
def run_one(self:Queue,
            job,             # a claimed job
            at:float=None    # the clock the retry delay is measured from; `took` is always real time
) -> AttrDict:
    'Dispatch one claimed job, then ack or fail it, and record the run either way.'
    t0 = now()
    # ack and fail are fenced, so a call that lost its lease changed nothing and the run says `lost`
    def _rep(out, status, error=None, result=None):
        if out['state'] == 'lost': status, error = 'lost', 'the lease was reclaimed mid-handler'
        return self._ran(job, status, t0, error, result)
    if (fn := self.handlers.get(job['kind'])) is None:
        err = f"no handler for {job['kind']!r}"
        return _rep(self.fail(job, err, at=at), 'error', err)
    try: res = fn(job['payload'])
    except Retry as e:
        return _rep(self.fail(job, f'Retry: {e}', after=e.after, at=at), 'retry', str(e))
    except Exception as e:
        err = f'{type(e).__name__}: {str(e)[:200]}'
        return _rep(self.fail(job, err, at=at), 'error', err)
    return _rep(self.ack(job, at=at), 'skipped' if isinstance(res, dict) and res.get('skipped') else 'ok',
                None, res)

@patch
def drain(self:Queue,
          worker:str='worker',  # lease holder recorded on each job
          limit:int=1000,       # jobs to run before returning; a ceiling, not a target
          batch:int=1,          # jobs claimed per round trip
          at:float=None
) -> L:
    'Claim and run until the queue is empty or `limit` jobs have run. This is the tick.'
    out = L()
    while len(out) < limit:
        got = self.claim(worker, n=min(batch, limit-len(out)), at=at)
        if not got: break
        out += got.map(lambda j: self.run_one(j, at=at))
    return out

## Looking at the queue

In [ ]:
#| export
@patch
def jobs(self:Queue, state:str=None, kind:str=None, limit:int=100) -> L:
    'Jobs, soonest first, filtered by state and kind.'
    w, a = [], []
    if state: w.append('state=?'); a.append(state)
    if kind:  w.append('kind=?');  a.append(kind)
    sql = 'SELECT * FROM jobs' + (' WHERE ' + ' AND '.join(w) if w else '') + ' ORDER BY run_at LIMIT ?'
    return self.q(sql, a + [limit])

@patch
def dead(self:Queue, limit:int=100) -> L:
    'Jobs that used up their attempts, with the error that finished them off.'
    return self.jobs(state='dead', limit=limit)

@patch
def pending(self:Queue, kind:str=None) -> int:
    'How many jobs are ready or running: what is left to do, counted rather than listed.'
    w, a = ('AND kind=? ', [kind]) if kind else ('', [])
    return self.one(f"SELECT count(*) n FROM jobs WHERE state IN ('ready','running') {w}", a)['n']

@patch
def get(self:Queue, job_id:str) -> AttrDict:
    'One job by id.'
    return self.one('SELECT * FROM jobs WHERE id=?', [job_id])

@patch
def retry(self:Queue, job_id:str, after:float=0, at:float=None) -> AttrDict:
    'Put a dead or pending job back on the queue, with its attempt count reset.'
    at = ifnone(at, now())
    with self.txn():
        self.db.q('UPDATE jobs SET state=?, attempt=0, run_at=?, error=NULL, worker=NULL, '
                  'lease_until=0, updated_at=? WHERE id=?', ['ready', at+after, at, job_id])
    return self.get(job_id)

@patch
def history(self:Queue, job_id:str=None, kind:str=None, limit:int=50) -> L:
    'Run history, newest first. Every row still holds what its handler gave back.'
    w, a = [], []
    if job_id: w.append('job_id=?'); a.append(job_id)
    if kind:   w.append('kind=?');   a.append(kind)
    sql = 'SELECT * FROM job_runs' + (' WHERE ' + ' AND '.join(w) if w else '')
    return self.runs.q(sql + ' ORDER BY started_at DESC LIMIT ?', a + [limit])

@patch
def stats(self:Queue) -> dict:
    'How many jobs in each state, and when the next one is due.'
    by = {r['state']: r['n'] for r in self.db.q('SELECT state, count(*) n FROM jobs GROUP BY state')}
    nxt = self.one("SELECT min(run_at) t FROM jobs WHERE state='ready'")
    return dict({s: by.get(s, 0) for s in STATES}, next_due=nxt['t'])

@patch
def purge(self:Queue, before:float=None) -> dict:
    'Delete finished jobs and run history older than `keep`. Dead jobs stay.'
    cut = ifnone(before, now() - self.keep)
    with self.txn():
        d = self.db.q("DELETE FROM jobs WHERE state='done' AND updated_at<? RETURNING id", [cut])
        r = self.db.q('DELETE FROM job_runs WHERE ended_at<? RETURNING id', [cut])
    return dict(jobs=len(d), runs=len(r))

One round trip, start to finish:

In [ ]:
from pobblebonk.core import vault

q = Queue(vault())
seen = []
q.register('greet', lambda p: {'said': f"hello {p['who']}"})
q.enqueue('greet', dict(who='milk'))
ran = q.drain('w1')
test_eq(len(ran), 1)
test_eq(ran[0].status, 'ok')
test_eq(ran[0].result, {'said': 'hello milk'})
test_eq(q.stats()['done'], 1)

The answer survives the run, which is the point:

In [ ]:
test_eq(q.history()[0]['result'], {'said': 'hello milk'})

An idempotency key means the same fire cannot be queued twice:

In [ ]:
a = q.enqueue('greet', dict(who='again'), key='wed-8pm')
b = q.enqueue('greet', dict(who='again'), key='wed-8pm')
test_eq(a.id, b.id)
test_eq(q.pending('greet'), 1)

A handler that raises is retried, and dead-letters once its attempts are spent:

In [ ]:
q2 = Queue(vault(), max_attempts=2, base=0)
q2.register('bad', lambda p: 1/0)
j = q2.enqueue('bad', {}, at=100)
test_eq(q2.drain('w1', at=100)[0].status, 'error')
test_eq(q2.get(j.id).state, 'ready')            # backed off, waiting for attempt two
test_eq(q2.drain('w1', at=100), [])             # `backoff` never answers zero, so not in this second
test_eq(q2.drain('w1', at=200)[0].status, 'error')
test_eq(q2.get(j.id).state, 'dead')
test_eq(len(q2.dead()), 1)

`Retry` is for work that did not raise on its own, and it can name its own delay:

In [ ]:
def wall(payload): raise Retry('a bot wall', after=300)

q3 = Queue(vault())
q3.register('wall', wall)
j3 = q3.enqueue('wall', {}, at=10)
test_eq(q3.drain('w1', at=10)[0].status, 'retry')
test_eq(q3.get(j3.id).run_at, 310)

Even a handler that asks to be retried at once waits a second, so one `drain` cannot spend every
attempt a job had:

In [ ]:
q3b = Queue(vault(), max_attempts=5)
q3b.register('wall', lambda p: (_ for _ in ()).throw(Retry('now please', after=0)))
q3b.enqueue('wall', {}, at=10)
test_eq(len(q3b.drain('w1', at=10)), 1)
test_eq(q3b.stats()['ready'], 1)

A lease that expires puts the job back, so a worker killed mid-job loses the claim and not the work:

In [ ]:
q4 = Queue(vault(), lease=0, base=0)
q4.register('slow', lambda p: 'done')
q4.enqueue('slow', {})
held = q4.claim('gone-away')
test_eq(len(held), 1)
test_eq(len(q4.reclaim()), 1)
test_eq(q4.stats()['ready'], 1)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()